# Community problem-shaped examples

Drive tilearray against **problem classes** that show up in public WMTS / WCS / WMS workflows — not one locked case per forum thread.

```bash
uv sync --dev
uv run jupyter lab   # open this notebook
```

Habit: `config → create_array → compute_mosaic → plot`.

Stay polite: use presets / modest bboxes; no bulk OSM scrape. Attribution where the host requires it.

| Section | Problem class | Host used here | Honest status today |
|---------|---------------|----------------|---------------------|
| 1 | Tile-grid AOI → georeferenced mosaic | NASA GIBS XYZ | Works on Mercator/slippy XYZ; native WMTS matrices also shipped (see docs/guides/wmts.md) |
| 2 | Numerical WCS AOI under size habits | EA Lidar WCS | Chunked fetch + clipped GetCoverage pad/align |
| 3 | Config / CRS mistakes | Offline checks | Typos fail early; any pyproj-valid CRS accepted |
| 4 | Polite multi-tile fetch + opt-in soft-fill | EA Lidar (small) | Progress + presets work; soft-fail is opt-in |

This notebook is for **feel and evidence**. Earlier open gaps (#92/#93/#95/#60/#87) have landed on main — keep driving the hosts, and file *new* gaps as issues.


## Setup

In [ ]:
from __future__ import annotations

import time

import matplotlib.pyplot as plt
import numpy as np

import tilearray
from tilearray import (
    WCSConfig,
    XYZConfig,
    compute_mosaic,
    create_array,
)
from tilearray.errors import ValidationError
from tilearray.fetch_presets import (
    EA_LIDAR_COVERAGE_ID,
    EA_LIDAR_WCS_URL,
    ea_dsp_fetch_defaults,
)
from tilearray.types import CRS, Format

print("tilearray", tilearray.__version__)


def summarize(arr, *, wall_s: float | None = None) -> None:
    if wall_s is not None:
        print(f"wall_s={wall_s:.2f}")
    print(f"shape={arr.shape}  dtype={arr.dtype}  dims={arr.dims}")
    for key in ("mosaic_crs", "service_type", "nodata", "band_policy", "transform_source"):
        if key in arr.attrs:
            print(f"  {key}={arr.attrs[key]!r}")


## 1 — Tile-grid AOI → georeferenced mosaic

**Job people ask for:** given a geographic bbox (and a zoom / matrix), get a spatially referenced array — not a pile of ungeoreferenced tile files.

**What we drive here:** NASA GIBS Blue Marble as XYZ (Mercator-compatible path). This is the *working* slice of the tile-grid job.

**Also shipped:** native projected WMTS TileMatrixSets (e.g. Kartverket topo EPSG:25833) — see `docs/guides/wmts.md`. This cell stays on the GIBS XYZ happy path so the notebook stays quick and polite.


In [ ]:
# Small mid-Atlantic window — keep crawls modest (GIBS terms).
GIBS_TEMPLATE = (
    "https://gibs.earthdata.nasa.gov/wmts/epsg3857/best/"
    "BlueMarble_NextGeneration/default/GoogleMapsCompatible_Level8/"
    "{z}/{y}/{x}.jpeg"
)

config = XYZConfig.from_url(
    GIBS_TEMPLATE,
    zoom=3,
    output_format=Format.JPEG,
    chunk_size=(256, 256),
    max_concurrent_requests=2,
    rate_limit_per_second=2.0,
    headers={
        "User-Agent": "tilearray (+https://github.com/MatthewJWhittle/tilearray/issues)",
    },
)

bbox = (-40.0, 20.0, -10.0, 45.0)  # lon/lat
t0 = time.perf_counter()
da = create_array(config, bbox, CRS.EPSG_4326)
arr = compute_mosaic(da)
summarize(arr, wall_s=time.perf_counter() - t0)

# Georef sanity: mosaic CRS + Affine transform from planned bbox (pixel centres).
assert arr.attrs.get("mosaic_crs") == CRS.EPSG_3857.value
assert "transform" in arr.attrs
print("transform=", arr.attrs["transform"])

fig, ax = plt.subplots(figsize=(7, 5))
# JPEG RGB → (y, x, band)
if arr.ndim == 3:
    ax.imshow(np.clip(arr.values / 255.0, 0, 1), origin="upper")
else:
    ax.imshow(arr.values, origin="upper", cmap="gray")
ax.set_title("GIBS Blue Marble — tile-grid AOI mosaic")
ax.set_axis_off()
plt.show()
print("Imagery: NASA GIBS / BlueMarble_NextGeneration")


## 2 — Numerical WCS AOI (chunk under server habits)

**Job people ask for:** pull analysis-ready elevation / coverage values for a bbox when servers enforce size or trim responses at edges.

**What we drive here:** a small EA Lidar DTM window with the polite DSP preset and explicit chunk/resolution planning.

**Also shipped:** clipped / partial GeoTIFF GetCoverage is placed on the request grid with NaN padding (no stretch). This cell shows the polite EA happy path.


In [ ]:
config = WCSConfig.from_url(
    EA_LIDAR_WCS_URL,
    EA_LIDAR_COVERAGE_ID,
    crs=CRS.EPSG_27700,
    output_format=Format.GEOTIFF,
    chunk_size=(64, 64),
    resolution=(1.0, 1.0),
    **ea_dsp_fetch_defaults(),
)

# ~128 m OSGB window — keep it small for interactive runs
bbox = (431900.0, 382700.0, 432028.0, 382828.0)

t0 = time.perf_counter()
da = create_array(config, bbox, CRS.EPSG_27700)
elevation = compute_mosaic(da)
summarize(elevation, wall_s=time.perf_counter() - t0)

print(
    f"mean={float(np.nanmean(elevation.values)):.2f} m  "
    f"nodata={elevation.attrs.get('nodata')!r}"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im = axes[0].imshow(elevation.values, origin="upper", cmap="terrain")
axes[0].set_title("EA Lidar DTM (OSGB)")
axes[0].set_axis_off()
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="m")
axes[1].hist(elevation.values.ravel(), bins=40, color="#4a6670")
axes[1].set_title("Elevation histogram")
axes[1].set_xlabel("m")
plt.tight_layout()
plt.show()


## 3 — Config / CRS mistakes fail early

**Job people ask for:** when a typo or unsupported CRS would otherwise become a misleading server error (classic `EPGS:` vs `EPSG:` footgun), the client should reject early with something actionable.

**What we drive here:** offline checks only — no network. Typos (`EPGS:…`) still fail early; any pyproj-valid CRS (including `EPSG:28992` / `EPSG:25833`) is accepted.


In [ ]:
# Typo class — should not silently rewrite EPGS → EPSG
try:
    CRS.from_string("EPGS:28992")
except ValidationError as exc:
    print("typo rejected:", exc)
except Exception as exc:  # enum path may raise ValueError today
    print(type(exc).__name__, ":", exc)

# Valid CRS codes — accepted via pyproj (#92)
for code in ("EPSG:28992", "EPSG:25833", "EPSG:32632"):
    print("ok:", CRS.from_string(code))

# Supported preset path
print("ok:", CRS.from_string("EPSG:27700"))

# Still invalid
try:
    CRS.from_string("EPSG:999999")
    print("EPSG:999999 unexpectedly accepted")
except ValidationError as exc:
    print("invalid rejected:", exc)


## 4 — Polite multi-tile fetch (+ optional soft-fill)

**Job people ask for:** larger mosaics without stampeding the host; visible progress; loud failure by default; opt-in partial fill when that is the right trade.

**What we drive here:** a slightly larger EA window under the DSP preset (ceiling 10), default progress, then a peek at `soft_fail=True` on a tiny grid. For OSM, prefer `osm_fetch_defaults()` and stay well within OSMF tile policy — never bulk-scrape.


In [ ]:
config = WCSConfig.from_url(
    EA_LIDAR_WCS_URL,
    EA_LIDAR_COVERAGE_ID,
    crs=CRS.EPSG_27700,
    output_format=Format.GEOTIFF,
    chunk_size=(64, 64),
    resolution=(1.0, 1.0),
    **ea_dsp_fetch_defaults(),
)

# ~256 m → a few tiles; progress bar should appear in notebook/TTY
bbox = (431900.0, 382700.0, 432156.0, 382956.0)

t0 = time.perf_counter()
da = create_array(config, bbox, CRS.EPSG_27700)  # show_progress defaults on
arr = compute_mosaic(da)
summarize(arr, wall_s=time.perf_counter() - t0)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(arr.values, origin="upper", cmap="terrain")
ax.set_title("Polite multi-tile EA mosaic")
ax.set_axis_off()
plt.show()


In [ ]:
# Soft-fill is opt-in. Default remains hard-fail after retries.
# This cell uses a normal successful mosaic with soft_fail enabled so you can
# inspect attrs; it does not intentionally break the host.

da_soft = create_array(
    config,
    (431900.0, 382700.0, 432028.0, 382828.0),
    CRS.EPSG_27700,
    soft_fail=True,
    soft_fail_max_fraction=0.25,
)
arr_soft = compute_mosaic(da_soft)
print("tilearray_tile_errors=", arr_soft.attrs.get("tilearray_tile_errors"))


## Multi-AOI tip (no orchestration engine)

For many small patches, an ordinary Python loop over bboxes is enough — pass a shared `cache_dir=` so overlapping source tiles warm-reuse. Recipe: `docs/multi-aoi-ml-patches.md` (no multi-AOI orchestrator).


In [ ]:
from pathlib import Path

cache = Path("/tmp/tilearray-community-examples-cache")
cache.mkdir(parents=True, exist_ok=True)

patches = [
    (431900.0, 382700.0, 431964.0, 382764.0),
    (432000.0, 382800.0, 432064.0, 382864.0),
]

means = []
for patch in patches:
    da_p = create_array(
        config,
        patch,
        CRS.EPSG_27700,
        cache_dir=cache,
        show_progress=False,
    )
    out = compute_mosaic(da_p)
    means.append(float(np.nanmean(out.values)))

print("patch means (m):", means)
print("cache entries:", len(list(cache.glob('*.tile'))))


## What to write down after a run

For each section: did the public API get you a **trusted** georeferenced array with little protocol ceremony? Note wall time, shape, CRS/transform, and anything confusing. Gaps you hit belong on the owning GitHub issues — not as silent workarounds in this notebook.
